In [1]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path("/Users/daniel/code/lca-lc-foundations/.env")
file_values = dotenv_values(env_path)

load_dotenv(env_path, override=True)

loaded_key = os.environ.get("LANGSMITH_API_KEY", "")
file_key = file_values.get("LANGSMITH_API_KEY", "")

print("Using endpoint:", os.environ.get("LANGSMITH_ENDPOINT"))
print("Key matches .env:", loaded_key == file_key)

Using endpoint: https://apac.api.smith.langchain.com
Key matches .env: True


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [3]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

In [5]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='26da6c56-6479-43fe-8319-7f4a778c1cc7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1626, 'prompt_tokens': 202, 'total_tokens': 1828, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1600, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYSXwAbOeLiwvqCppzzTQJil7Rlz8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13477-3716-7571-b664-c849ede4d533-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_WnUHIPlkCFpCqFJp8DX8cDaa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me